In [1]:
import os, json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                                   # reads .env; the key never appears in the notebook
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ.get("OPENAI_BASE_URL"))
MODEL = "gpt-4o-mini"                            # use the model your course key allows

r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Reply with the single word: ready"}])
print(r.choices[0].message.content)

ready


## 1. Data heading

In [2]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

TEST2 = Path("data/ims/2nd_test")
files = sorted(p for p in TEST2.iterdir() if p.is_file())
print(len(files), "snapshots:", files[0].name, "→", files[-1].name)

x = np.loadtxt(files[0])                       # one snapshot: rows = samples, columns = bearings
print("shape:", x.shape, "dtype:", x.dtype)
print(pd.DataFrame(x, columns=[f"b{i+1}" for i in range(x.shape[1])]).describe().round(3))

984 snapshots: 2004.02.12.10.32.39 → 2004.02.19.06.22.39
shape: (20480, 4) dtype: float64
              b1         b2         b3         b4
count  20480.000  20480.000  20480.000  20480.000
mean      -0.010     -0.013     -0.015     -0.010
std        0.073      0.090      0.108      0.053
min       -0.386     -0.513     -0.911     -0.264
25%       -0.059     -0.073     -0.081     -0.046
50%       -0.010     -0.015     -0.015     -0.010
75%        0.037      0.046      0.051      0.027
max        0.454      0.464      1.023      0.193


## 2. Compute health indicators across the whole test

In [3]:
from scipy.signal import butter, sosfiltfilt, hilbert
FS, RPM = 20000, 2000
fr = RPM / 60
n, D, d, phi = 16, 2.815, 0.331, np.deg2rad(15.17)   # ZA-2115 geometry (Qiu et al., 2006) — verify
r = (d / D) * np.cos(phi)
FAULT_HZ = {"BPFO": n / 2 * fr * (1 - r), "BPFI": n / 2 * fr * (1 + r),
            "BSF": D / (2 * d) * fr * (1 - r ** 2), "FTF": fr / 2 * (1 - r)}
print({k: round(v, 1) for k, v in FAULT_HZ.items()})

SOS = butter(4, [1000, 9000], btype="band", fs=FS, output="sos")   # band-pass before the envelope

def snapshot_features(x):
    """Health indicators for one bearing's 1-s snapshot: RMS, kurtosis, crest factor and
    envelope-spectrum amplitude at each fault frequency (relative to the median envelope level)."""
    env = np.abs(hilbert(sosfiltfilt(SOS, x)))
    spec = np.abs(np.fft.rfft(env - env.mean())) / len(env)
    f = np.fft.rfftfreq(len(env), 1 / FS)
    base = np.median(spec[(f > 5) & (f < 500)])
    feats = {"rms": np.sqrt(np.mean(x ** 2)),
             "kurtosis": pd.Series(x).kurt(),
             "crest": np.max(np.abs(x)) / np.sqrt(np.mean(x ** 2))}
    for k, hz in FAULT_HZ.items():
        feats[f"env_{k}"] = spec[(f > hz - 2) & (f < hz + 2)].max() / base
    return feats

rows = []
for i, p in enumerate(files):
    sig = np.loadtxt(p)
    t = pd.to_datetime(p.name, format="%Y.%m.%d.%H.%M.%S")
    for b in range(4):
        rows.append({"idx": i, "time": t, "bearing": b + 1, **snapshot_features(sig[:, b])})
feat = pd.DataFrame(rows)
os.makedirs("results", exist_ok=True)
feat.to_csv("results/test2_features.csv", index=False)   # cached: the agent reads this, not the raw files
feat.groupby("bearing")[["rms", "kurtosis", "env_BPFO"]].describe().round(2)

{'BPFO': np.float64(236.4), 'BPFI': np.float64(296.9), 'BSF': np.float64(139.9), 'FTF': np.float64(14.8)}


rms                                          kurtosis        ...  \
         count  mean   std  min   25%   50%   75%   max    count  mean  ...   
bearing                                                                 ...   
1        984.0  0.11  0.06  0.0  0.08  0.08  0.11  0.73    984.0  0.83  ...   
2        984.0  0.10  0.01  0.0  0.09  0.09  0.10  0.22    984.0  0.15  ...   
3        984.0  0.10  0.01  0.0  0.10  0.10  0.10  0.19    984.0  1.33  ...   
4        984.0  0.06  0.01  0.0  0.06  0.06  0.06  0.15    984.0  0.14  ...   

                     env_BPFO                                             
          75%    max    count  mean   std   min   25%   50%   75%    max  
bearing                                                                   
1        0.88  14.11    984.0  2.06  0.98  0.49  1.42  1.79  2.40  11.11  
2        0.21   2.45    984.0  1.54  0.58  0.43  1.19  1.48  1.80  10.40  
3        1.65   3.70    984.0  2.04  0.62  0.54  1.63  1.97  2.39   7.32  
4        0.18   1.65    984.0  1.71  0.88  0.48  1.33  1.64  1.97  19.64  

[4 rows x 24 columns]